In [12]:
!wget -O lesson_rag_helper.py https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-21 11:05:59--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘lesson_rag_helper.py’

lesson_rag_helper.p 100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-21 11:06:00 (10.3 MB/s) - ‘lesson_rag_helper.py’ saved [2134/2134]



In [46]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index
from lesson_rag_helper import RAGBase
from dataclasses import dataclass
from openai import OpenAI
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools

In [2]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [3]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

How many lesson pages are there?

In [4]:
len(documents)

72

In [7]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

Searching with this query. What is the first result?

>How does the agentic loop keep calling the model until it stops?

In [10]:
query = "How does the agentic loop keep calling the model until it stops?"

results = index.search(query, num_results=5)

results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

In [16]:
@dataclass
class RAGResponse:
    answer: str
    usage: object


class MyRAG(RAGBase):

    def search(self, query: str, num_results: int = 5) -> list:
        return self.index.search(query, num_results=num_results)

    def build_context(self, search_results: list) -> str:
        lines = []
        for doc in search_results:
            lines.append(f"File: {doc['filename']}")
            lines.append(doc['content'])
            lines.append('---')
        return '\n'.join(lines).strip()

    def llm(self, prompt: str) -> tuple:
        messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt},
        ]
        response = self.llm_client.responses.create(
            model=self.model,
            input=messages,
        )
        return response.output_text, response.usage

    def rag(self, query: str) -> RAGResponse:
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer, usage = self.llm(prompt)
        return RAGResponse(answer=answer, usage=usage)

In [21]:
client = OpenAI()

my_rag = MyRAG(index=index, llm_client=client, model="gpt-4o-mini")

How many input (prompt) tokens are sent to the model for the following request?

> How does the agentic loop keep calling the model until it stops?

In [25]:
result = my_rag.rag("How does the agentic loop keep calling the model until it stops?")
print(result.answer)
print("\nInput tokens: " + str(result.usage.input_tokens))

The agentic loop continues to call the model until it stops by utilizing a `while` loop that checks for function calls in the model's responses. Here's how it works:

1. **Initialization**: The process starts with initial messages, including developer instructions and a user question.

2. **API Call Loop**: Inside a `while True` loop, the model is called repeatedly using the `openai_client.responses.create()` function. This sends the current message history to the model.

3. **Processing Responses**: After receiving a response:
   - Each entry is checked. If the entry is a `function_call`, the loop identifies that the model needs to perform an action (like a search), and executes that action.
   - The result of the function call is then appended to the message history.

4. **Exit Condition**: The loop includes a flag (`has_function_calls`) that is set to `True` when the model specifies a function call. If no function calls are present in the model's responses during an iteration, the l

After implementing chunking with size 2000 and step 1000, how many chunks do we have in the index?

In [28]:
chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [29]:
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)

In [30]:
chunk_rag = MyRAG(index=chunk_index, llm_client=client, model="gpt-4o-mini")

In [31]:
result_chunked = chunk_rag.rag("How does the agentic loop keep calling the model until it stops?")

print(result_chunked.usage.input_tokens)

2315


How much does chunking reduce the amount of input tokens?

In [35]:
print(f"Q3 tokens: {result.usage.input_tokens}")
print(f"Q5 tokens: {result_chunked.usage.input_tokens}")
print(f"Ratio: {result.usage.input_tokens / result_chunked.usage.input_tokens:.1f}x fewer")

Q3 tokens: 7132
Q5 tokens: 2315
Ratio: 3.1x fewer


In [54]:
# Define the search tool
def search(query: str) -> list:
    """Search the course lessons for information relevant to the query."""
    return chunk_index.search(query, num_results=5)

In [55]:
# Wrap the tool
tools = Tools()
tools.add_tool(search)

In [56]:
# Wrap the llm client
llm_client = OpenAIClient(model="gpt-4o-mini", client=client)

In [57]:
# Build the runner
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

runner = OpenAIResponsesRunner(
    tools=tools,
    developer_prompt=instructions,
    llm_client=llm_client
)

In [60]:
# Run it
result = runner.loop("How does the agentic loop work, and how is it different from plain RAG?")

In [61]:
search_calls = [
    m for m in result.all_messages
    if hasattr(m, 'name') and m.name == 'search'
]

print(f"Search was called {len(search_calls)} times")

Search was called 3 times
